# Visualize Stimulus Pairs
One example stimulus for every pair in the training and test datasets.

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms

from TransitiveTrainDataset import TransitiveTrainDataset
from TransitiveTestDataset import TransitiveTestDataset
from TransitiveTrainDataset_Exp import TransitiveTrainDataset_Exp

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST('./data', train=False, transform=transform)

In [ ]:
def unnormalize(img_tensor, mean=0.1307, std=0.3081):
    """Undo the normalization for display."""
    return img_tensor * std + mean


def plot_pair_examples(dataset, title="Stimulus Pairs", exception_pair=None):
    """Plot one example stimulus for each pair in the dataset."""
    n_pairs = len(dataset.pairs)
    cols = min(4, n_pairs)
    rows = (n_pairs + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    
    for pair_idx, pair in enumerate(dataset.pairs):
        # Get the first sample for this pair (winner on left)
        # Each pair has samples_per_pair entries in self.samples
        # Find the first sample matching this pair
        sample_found = False
        for s_idx, sample in enumerate(dataset.samples):
            if sample[0] == pair[0] and sample[1] == pair[1]:
                winner_digit, loser_digit, winner_img_idx, loser_img_idx = sample
                sample_found = True
                break
        
        if not sample_found:
            continue
        
        # Get the stimulus (winner on left)
        img_left = dataset.mnist[winner_img_idx][0]
        img_right = dataset.mnist[loser_img_idx][0]
        stimulus = torch.cat([img_left, img_right], dim=-1)
        
        # Unnormalize and plot
        img = unnormalize(stimulus).squeeze().numpy()
        
        ax = axes[pair_idx]
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        
        is_exception = exception_pair and (pair[0], pair[1]) == exception_pair
        color = 'magenta' if is_exception else 'black'
        label = f'{pair[0]} > {pair[1]}'
        if is_exception:
            label += ' (EXC)'
        ax.set_title(label, fontsize=12, fontweight='bold' if is_exception else 'normal', color=color)
        
        # Draw dividing line between left and right images
        ax.axvline(x=27.5, color='red', linewidth=1, linestyle='--')
        ax.set_xticks([])
        ax.set_yticks([])
    
    # Hide unused axes
    for i in range(n_pairs, len(axes)):
        axes[i].axis('off')
    
    fig.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return fig

## Training Pairs (adjacent)

In [ ]:
train_dataset = TransitiveTrainDataset(mnist_train, n=8, seed=42)
print(f"Pairs: {train_dataset.pairs}")
print(f"Total samples: {len(train_dataset.samples)}")
print(f"Total examples (with flips): {len(train_dataset)}")

fig = plot_pair_examples(train_dataset, title="Training Pairs (adjacent)")

## Test Pairs (non-adjacent, distance >= 2)

In [ ]:
test_dataset = TransitiveTestDataset(mnist_test, n=8, seed=42)
print(f"Pairs: {test_dataset.pairs}")
print(f"Total samples: {len(test_dataset.samples)}")
print(f"Total examples (with flips): {len(test_dataset)}")

fig = plot_pair_examples(test_dataset, title="Test Pairs (non-adjacent)")

## Training Pairs with Exception (adjacent + exception)

In [ ]:
exception_pair = (5, 2)
train_dataset_exp = TransitiveTrainDataset_Exp(mnist_train, n=8, seed=42, exception_pair=exception_pair)
print(f"Pairs: {train_dataset_exp.pairs}")
print(f"Total samples: {len(train_dataset_exp.samples)}")
print(f"Total examples (with flips): {len(train_dataset_exp)}")

fig = plot_pair_examples(train_dataset_exp, title="Training Pairs with Exception", exception_pair=exception_pair)

## Sample Details
Print the first few samples from each pair to verify the stored indices.

In [ ]:
print("Training dataset (base) — first 3 samples per pair:")
print(f"{'Pair':>10} | {'Winner digit':>12} | {'Loser digit':>11} | {'Winner MNIST idx':>16} | {'Loser MNIST idx':>15}")
print("-" * 75)

for pair in train_dataset.pairs:
    count = 0
    for sample in train_dataset.samples:
        if sample[0] == pair[0] and sample[1] == pair[1]:
            print(f"  {sample[0]} > {sample[1]}    |           {sample[0]} |          {sample[1]} |            {sample[2]:5d} |           {sample[3]:5d}")
            count += 1
            if count >= 3:
                break
    print()